# 01 Data Generation & Validation

This notebook documents how the synthetic fashion launch dataset is generated and checks that it behaves like a realistic execution portfolio. The data is fictional, but the structure reflects common launch dependencies across design, sourcing, production, quality, logistics, e-commerce, retail readiness, and launch governance.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from generate_data import main as generate_data
from transform_data import main as transform_data
from utils import load_all

generate_data()
transform_data()
data = load_all(processed=True)
{name: df.shape for name, df in data.items()}

## Table Coverage

The project creates the eight requested tables: brands, collections, suppliers, purchase orders, milestones, risks, incidents, and mitigation actions. Volumes are intentionally large enough for portfolio analysis without turning the repository into a data dump.

In [ ]:
pd.DataFrame({
    'table': list(data.keys()),
    'rows': [len(df) for df in data.values()],
    'columns': [df.shape[1] for df in data.values()],
})

## Date Consistency Checks

Launch milestones should be planned before launch. Purchase orders should not have impossible ordering and shipping sequences. Actual dates are allowed to be missing for open work, which mirrors live project tracking.

In [ ]:
collections = data['collections']
milestones = data['milestones']
pos = data['purchase_orders']

checks = {
    'milestones_planned_before_launch': (milestones['planned_date'] <= milestones['launch_date']).mean(),
    'po_order_before_ship': (pos['planned_order_date'] <= pos['planned_ship_date']).mean(),
    'po_ship_before_arrival': (pos['planned_ship_date'] <= pos['planned_arrival_date']).mean(),
    'open_milestones_with_missing_actual_dates': milestones.loc[milestones['status'] != 'Completed', 'actual_date'].isna().mean(),
}
checks

## Dependency and Delay Sanity

Milestones are generated in sequence. Some dependencies become blocked or delayed, increasing downstream risk and creating the kind of cross-workstream pressure a PMO needs to see.

In [ ]:
milestones.groupby('status').size().sort_values(ascending=False), milestones['delay_days'].describe()